[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/03b_surrogate_model.ipynb)

# 03b — Surrogate model: an FNO ⊕ WNO hybrid with FiLM conditioning

**Purpose.** Replace the ray tracer inside the optimizer loop. One evaluation
costs ~30–40 s of Sionna-RT today; the budget here is **under 5 s**.

**Inputs.** `data/interim/tilt_sweep.npz` and `data/interim/scene_features.npz`
from notebook 03a.

**Outputs.** `outputs/surrogate/operator.pt`, and the figures and tables under
`reports/*/03b_surrogate/`.

---

### The architecture

Per `(transmitter, band)` slice, a residual over the analytic pattern
re-embedding:

```
rsrp_after = rsrp_before + ΔA + residual        where the coverage head says a path exists
```

The trunk is four hybrid blocks, each summing three paths and modulating the
result:

- a **factorized Fourier layer** (Li et al., ICLR 2021; F-FNO, Tran et al.,
  ICLR 2023) for global structure,
- a **3-level Haar wavelet layer** (Tripura and Chakraborty, CMAME 2023) for
  local structure,
- a pointwise convolution,
- **FiLM** (Perez et al., AAAI 2018) conditioning on both tilts, their
  difference and the band — spatial as well as global, because tilting changes
  gain as a function of each tile's *own* elevation angle and one affine pair
  for the whole plane cannot say that.

Predicting a residual rather than a map is the load-bearing choice: a zero
output is the analytic answer, so an untrained or out-of-distribution forward
pass degrades to physics rather than to noise.

### What "performs well" has to mean

dB error is not the acceptance criterion. **Agreeing with the ray tracer about
which tilt vector is better** is — that is the only thing the optimizer consumes.
Section 6 measures rank agreement and lexicographic-winner agreement against the
exact ground truth 03a made free.

## 0. Environment

Run this section first, wherever you are.

**Locally** it walks up to the project root and makes it the working directory,
so the root-relative paths in the configs resolve the way they do for
`task surrogate:dataset` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path`, and installs
what Colab does not ship. Training runs on a GPU if there is one and falls back to CPU
if not.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook in this project.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab ships numpy, pandas, pyarrow, matplotlib and
# seaborn; these are the ones it does not. sionna-rt needs a CUDA runtime, so
# select a GPU runtime before running this notebook.
COLAB_PACKAGES = [
    ("hydra", "hydra-core"),
    ("sionna.rt", "sionna-rt"),
    ("ax", "ax-platform"),
]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR if SUBDIR else checkout
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Reading them
# from the environment lets a smoke run shrink the budget, or a Colab session
# repoint the paths, without editing the notebook.
CONFIG_OVERRIDES: list[str] = os.environ.get("BAND_TILT_OVERRIDES", "").split()

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ is DVC-tracked and not part of the clone, so a fresh Colab runtime has
# no scenario to optimize against. Either run notebook 00 first, or mount Drive
# and point the config at a copy that already holds one. Drive also survives a
# runtime reset, which /content does not.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# PROJECT_ROOT = "/content/drive/MyDrive/band-tilt"
# CONFIG_OVERRIDES += [
#     f"simulation.output.manifest_file={PROJECT_ROOT}/data/external/scenario.json",
#     f"data.output.mdt_file={PROJECT_ROOT}/data/processed/mdt.parquet",
#     f"optim.output.dir={PROJECT_ROOT}/outputs/optim",
# ]

## 1. Setup

The same composed config the pipeline stage reads, and the artifacts 03a wrote.

In [ ]:
import time
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.config import load_config
from src.evaluation.export import save_table
from src.optim.objective import KPI_NAMES, MAXIMISED, KpiVector, lexicographic_best
from src.optim.space import TiltSpace
from src.surrogate import dataset as surrogate_data
from src.surrogate.evaluator import SurrogateEvaluator
from src.surrogate.model import OperatorSpec, TiltOperator
from src.surrogate.train import TrainSpec, fit
from src.utils.plotting import save_fig, setup_plotting

cfg = load_config(CONFIG_OVERRIDES)
setup_plotting()
save_fig = partial(save_fig, in_colab=IN_COLAB, directory=Path("reports/figures/03b_surrogate"))
save_table = partial(save_table, in_colab=IN_COLAB, directory=Path("reports/tables/03b_surrogate"))

space = TiltSpace.from_config(cfg)
sweep = surrogate_data.Sweep.load(cfg.surrogate.output.sweep_file)
features = surrogate_data.load_features(cfg.surrogate.output.feature_file)
pattern = surrogate_data.fit_pattern(sweep, features.elevation_deg)

train_pairs = surrogate_data.TiltPairs(sweep, features, pattern, space.cells, "train")
test_pairs = surrogate_data.TiltPairs(sweep, features, pattern, space.cells, "test")
print(f"train {len(train_pairs)} pairs   test {len(test_pairs)} pairs")
print(f"device available: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. The operator

Small on purpose. The binding constraint is a few thousand training pairs, not
capacity — so the spectral layer is factorized rather than dense. A dense
`[width, width, modes, modes]` weight, the standard FNO form, would be about
4.2 M parameters and **99% of the model**; factorizing it into a per-mode
diagonal and one channel mixing costs a fraction of that at no reported
accuracy cost.

In [ ]:
operator = OperatorSpec(
    in_channels=len(surrogate_data.TiltPairs.channels),
    cond_dim=3 + len(sweep.band_names),
    **{field: cfg.surrogate.model[field] for field in cfg.surrogate.model},
)
train_spec = TrainSpec.from_config(cfg)

print(operator)
print(f"\nparameters: {TiltOperator(operator).n_parameters:,}")
print(f"\n{train_spec}")

## 3. Train

Three terms. `level` is Huber on dB wherever the **target** has a path — not
wherever both ends do, which would drop exactly the tiles a tilt change opened
and leave the coverage head's own tiles unsupervised. `coverage` is
cross-entropy on the path mask. `cycle` requires the operator run backwards on
its own prediction to return the map it started from: free supervision, and a
physical identity rather than a penalty picked for convenience.

In [ ]:
model, history = fit(train_pairs, test_pairs, train_spec, operator)

curve = pd.DataFrame([vars(report) for report in history])
save_table(curve, "training_curve")
curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

axes[0].plot(curve["epoch"], curve["train_mae_db"], marker="o", label="train")
axes[0].plot(curve["epoch"], curve["test_mae_db"], marker="o", label="test (unseen tilts)")
axes[0].set(xlabel="epoch", ylabel="mean absolute error (dB)", title="Level error")
axes[0].legend()

axes[1].plot(curve["epoch"], curve["test_coverage_f1"], marker="o", color="C2")
axes[1].set(xlabel="epoch", ylabel="F1", title="Coverage head, test split")

fig.tight_layout()
save_fig(fig, "training_curve")

In [ ]:
checkpoint = Path(cfg.surrogate.output.model_file)
checkpoint.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "state_dict": model.state_dict(),
        "operator": vars(operator),
        "train": vars(train_spec),
        "history": [vars(report) for report in history],
        "sweep_file": str(cfg.surrogate.output.sweep_file),
        "feature_file": str(cfg.surrogate.output.feature_file),
    },
    checkpoint,
)
print(f"{checkpoint}  {checkpoint.stat().st_size / 1e6:.1f} MB")

## 4. Per-tile error against the ray tracer

Split by what the tile is worth. A 3 dB error on a tile at −70 dBm changes no
KPI; the same error at the −120 dBm hole threshold changes two.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
surrogate = SurrogateEvaluator(cfg=cfg, encoder=train_pairs.encoder, model=model,
                               device=DEVICE, keep_rsrp=True)
analytic = SurrogateEvaluator(cfg=cfg, encoder=train_pairs.encoder, model=model,
                              device=DEVICE, keep_rsrp=True, analytic_only=True)

rng = np.random.default_rng(int(cfg.seed))


def random_tilt_vectors(count: int) -> np.ndarray:
    """Tilt vectors drawn on the sweep's own grid, so ground truth is exact."""
    n_band = len(sweep.band_names)
    drawn = np.empty((count, space.n_dim))
    for dimension in range(space.n_dim):
        grid = sweep.tilts[dimension % n_band]
        drawn[:, dimension] = rng.choice(grid, size=count)
    return drawn


PROBES = random_tilt_vectors(200)
truth = np.stack([sweep.at(vector) for vector in PROBES[:40]])
predicted = np.stack([surrogate.predict(vector) for vector in PROBES[:40]])
baseline = np.stack([analytic.predict(vector) for vector in PROBES[:40]])
print(f"compared {len(truth)} maps of shape {truth.shape[1:]}")

In [ ]:
hole_dbm = float(cfg.kpi.hole_dbm)
weak_dbm = float(cfg.kpi.weak_dbm)
both = np.isfinite(truth) & np.isfinite(predicted)

rows = []
for label, low, high in [
    ("served (> weak)", weak_dbm, np.inf),
    ("weak", hole_dbm, weak_dbm),
    ("near hole (<= hole)", -np.inf, hole_dbm),
]:
    band = both & (truth > low) & (truth <= high)
    for name, other in [("hybrid", predicted), ("analytic", baseline)]:
        error = np.abs(other[band] - truth[band]) if band.any() else np.array([np.nan])
        rows.append({
            "tiles": label,
            "model": name,
            "n": int(band.sum()),
            "mae_db": float(np.nanmean(error)),
            "p95_db": float(np.nanpercentile(error, 95)),
        })

coverage_truth = np.isfinite(truth)
for name, other in [("hybrid", predicted), ("analytic", baseline)]:
    hit = np.isfinite(other)
    rows.append({
        "tiles": "coverage (F1)",
        "model": name,
        "n": int(coverage_truth.size),
        "mae_db": float(2 * (hit & coverage_truth).sum() / (hit.sum() + coverage_truth.sum())),
        "p95_db": np.nan,
    })

tile_errors = pd.DataFrame(rows)
save_table(tile_errors, "tile_error")
tile_errors

## 5. Wall clock

The reason any of this exists.

In [ ]:
timings = []
for name, evaluator in [("hybrid", surrogate), ("analytic", analytic)]:
    evaluator.predict(PROBES[0])  # warm the kernels; the first call is not the cost
    started = time.perf_counter()
    for vector in PROBES[:20]:
        evaluator.evaluate(vector)
    timings.append({"model": name, "seconds_per_evaluation": (time.perf_counter() - started) / 20})

timing = pd.DataFrame(timings)
timing["speedup_vs_ray_tracing"] = 35.0 / timing["seconds_per_evaluation"]
save_table(timing, "wall_clock")
print("ray tracing costs roughly 35 s per evaluation on this scenario; budget is 5 s\n")
timing

## 6. The measurement that decides it

The optimizer never sees a radio map. It sees five KPIs, and it only ever
compares them. So what matters is whether the surrogate **orders** tilt vectors
the way the ray tracer does.

Ground truth here costs nothing: `sweep.at(vector)` returns the exact map for
any tilt vector on the sweep grid, because tilts decompose. Every number below
is measured against the real solver, on tilts the model was never trained on.

Two baselines, both free:

- **identity** — return the anchor map unchanged. Bounds how much tilt matters
  at all, so a good-looking correlation cannot be credited to the model when it
  belongs to the problem. It scores every candidate identically, so it can rank
  nothing and picks the first — which is the point it is making.
- **analytic ΔA** — the pattern re-embedding, zero learned parameters. If the
  hybrid does not beat this, that is the finding and it gets reported as one.

In [ ]:
from scipy import stats

from src.optim.objective import evaluate_kpis

mdt = pd.read_parquet(cfg.data.output.mdt_file)
anchor = sweep.at(space.baseline)


def kpis_for(maps: np.ndarray) -> KpiVector:
    return evaluate_kpis(maps, sweep.band_names, mdt, cfg)


vectors = {"truth": [], "hybrid": [], "analytic": [], "identity": []}
for vector in PROBES:
    vectors["truth"].append(kpis_for(sweep.at(vector)))
    vectors["hybrid"].append(kpis_for(surrogate.predict(vector)))
    vectors["analytic"].append(kpis_for(analytic.predict(vector)))
    vectors["identity"].append(kpis_for(anchor))

# KpiVector objects for lexicographic_best, which compares them with the KPI
# tolerances; plain arrays for the correlations.
scores = {name: np.stack([kpi.as_array() for kpi in values]) for name, values in vectors.items()}
print(f"scored {len(PROBES)} held-out tilt vectors through the real KPI functions")

In [ ]:
tolerance = np.array([float(cfg.kpi.tolerance[name]) for name in KPI_NAMES])

rows = []
for name in ("hybrid", "analytic", "identity"):
    for index, kpi in enumerate(KPI_NAMES):
        actual, other = scores["truth"][:, index], scores[name][:, index]
        rows.append({
            "model": name,
            "kpi": kpi,
            "direction": "max" if kpi in MAXIMISED else "min",
            "spearman": float(stats.spearmanr(actual, other).statistic),
            "mae": float(np.mean(np.abs(other - actual))),
            "tolerance": tolerance[index],
            "within_tolerance": float(np.mean(np.abs(other - actual) <= tolerance[index])),
        })

agreement = pd.DataFrame(rows)
save_table(agreement, "kpi_agreement")
agreement.pivot(index="kpi", columns="model", values="spearman")

In [ ]:
# The decision the optimizer actually makes: which candidate wins, under the
# same ADR 0001 lexicographic rule the searches use.
truth_winner = lexicographic_best(vectors["truth"], cfg)
verdict = []
for name in ("hybrid", "analytic", "identity"):
    winner = lexicographic_best(vectors[name], cfg)
    regret = scores["truth"][winner] - scores["truth"][truth_winner]
    verdict.append({
        "model": name,
        "picked_the_true_winner": bool(winner == truth_winner),
        **{f"regret_{kpi}": float(value) for kpi, value in zip(KPI_NAMES, regret, strict=True)},
    })

winners = pd.DataFrame(verdict)
save_table(winners, "lexicographic_winner")
winners

In [ ]:
fig, axes = plt.subplots(1, len(KPI_NAMES), figsize=(4 * len(KPI_NAMES), 3.8))
for ax, index in zip(axes, range(len(KPI_NAMES)), strict=True):
    kpi = KPI_NAMES[index]
    for name, colour in [("hybrid", "C0"), ("analytic", "C1")]:
        ax.scatter(scores["truth"][:, index], scores[name][:, index], s=9, alpha=0.5,
                   color=colour, label=name)
    limits = [scores["truth"][:, index].min(), scores["truth"][:, index].max()]
    ax.plot(limits, limits, color="0.4", linewidth=0.9, zorder=0)
    ax.set(xlabel="ray traced", ylabel="predicted", title=kpi)
axes[0].legend()

fig.suptitle("Surrogate against the ray tracer, on tilts it never saw")
fig.tight_layout()
save_fig(fig, "kpi_agreement")

## 7. Ablation: is the hybrid earning its parts?

The branches are separate, so switching one off is a config flag rather than a
second model. This is what turns "FNO and WNO help" from an assertion into a
measurement — and a negative result here is a result, not a failure.

Short runs, so read the ordering rather than the absolute numbers.

In [ ]:
ABLATION_EPOCHS = max(1, train_spec.epochs // 3)

rows = []
for label, changes in [
    ("full hybrid", {}),
    ("no FNO", {"use_fno": False}),
    ("no WNO", {"use_wno": False}),
    ("global FiLM only", {"use_spatial_film": False}),
    ("neither operator", {"use_fno": False, "use_wno": False}),
]:
    variant = OperatorSpec(**{**vars(operator), **changes})
    trained, curve_ = fit(
        train_pairs, test_pairs,
        TrainSpec(**{**vars(train_spec), "epochs": ABLATION_EPOCHS}),
        variant,
    )
    rows.append({
        "variant": label,
        "parameters": TiltOperator(variant).n_parameters,
        "test_mae_db": curve_[-1].test_mae_db,
        "test_coverage_f1": curve_[-1].test_coverage_f1,
        "seconds_per_epoch": float(np.mean([report.seconds for report in curve_])),
    })

ablation = pd.DataFrame(rows).sort_values("test_mae_db")
save_table(ablation, "ablation")
ablation

## 8. Handover

`SurrogateEvaluator` satisfies `src.optim.evaluator.ObjectiveEvaluator`, so any
search in `src/optim/methods/` takes it in place of the ray-traced `Evaluator`
without a line changing. The cell below is the proof: the same `run_search`
call, a different evaluator.

Read section 6 before using it for anything. If rank correlation is low or the
lexicographic winner disagrees, the surrogate is fast and wrong, which is worse
than slow and right.

In [ ]:
from src.optim.methods import run_search

# The method comes from the config group, not an argument -- selecting a method
# selects its budget with it, so the two cannot be mismatched.
random_cfg = load_config([*CONFIG_OVERRIDES, "optim/method=random"])

with SurrogateEvaluator(cfg=cfg, encoder=train_pairs.encoder, model=model, device=DEVICE) as fast:
    started = time.perf_counter()
    history_ = run_search(fast, random_cfg)
    elapsed = time.perf_counter() - started

print(f"{fast.n_calls} evaluations in {elapsed:.1f} s "
      f"({elapsed / max(fast.n_calls, 1):.3f} s each)")
print(f"ray tracing the same budget would have cost about {fast.n_calls * 35 / 3600:.1f} hours")